In [3]:
"""
Simple BPE Tokenizer Implementation
Educational implementation to understand how tokenization works in LLMs
"""

import numpy as np
from collections import Counter


class SimpleBPETokenizer:
    """
    A minimal Byte Pair Encoding tokenizer for learning purposes.
    
    This implements the core BPE algorithm:
    1. Start with character-level tokens
    2. Iteratively merge the most frequent pair
    3. Build vocabulary of learned subword units
    """
    
    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size
        self.vocab = {}  # token_id -> token_string
        self.merges = {}  # (token1, token2) -> merged_token_id
        self.char_to_id = {}
        self.id_to_char = {}
        
    def _get_stats(self, tokens):
        """
        Count frequency of adjacent token pairs.
        
        Args:
            tokens: List of token IDs
            
        Returns:
            Counter object mapping (token1, token2) -> frequency
        """
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += 1
        return pairs
    
    def _merge_pair(self, tokens, pair, new_token_id):
        """
        Replace all occurrences of a token pair with a new merged token.
        
        Args:
            tokens: List of token IDs
            pair: Tuple of (token1_id, token2_id) to merge
            new_token_id: ID for the merged token
            
        Returns:
            New list of tokens with pairs merged
        """
        new_tokens = []
        i = 0
        while i < len(tokens):
            # Check if current position matches the pair
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_token_id)
                i += 2  # Skip both tokens in the pair
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens
    
    def train(self, text, verbose=True):
        """
        Learn BPE vocabulary from training text.
        
        Args:
            text: String to train on
            verbose: Whether to print progress
        """
        # Step 1: Initialize with character-level tokens
        unique_chars = sorted(set(text))
        self.char_to_id = {ch: i for i, ch in enumerate(unique_chars)}
        self.id_to_char = {i: ch for i, ch in enumerate(unique_chars)}
        
        # Convert text to initial token sequence (all characters)
        tokens = [self.char_to_id[ch] for ch in text]
        
        # Start with character vocabulary
        self.vocab = self.id_to_char.copy()
        
        num_merges = self.vocab_size - len(unique_chars)
        
        if verbose:
            print(f"{'='*60}")
            print(f"Training BPE Tokenizer")
            print(f"{'='*60}")
            print(f"Starting vocabulary: {len(unique_chars)} characters")
            print(f"Target vocabulary: {self.vocab_size} tokens")
            print(f"Will perform {num_merges} merges")
            print(f"Text length: {len(text):,} characters → {len(tokens):,} initial tokens\n")
        
        # Step 2: Iteratively merge most frequent pairs
        for merge_num in range(num_merges):
            # Count pair frequencies
            stats = self._get_stats(tokens)
            
            if not stats:
                if verbose:
                    print(f"\nNo more pairs to merge after {merge_num} merges")
                break
            
            # Find most frequent pair
            most_frequent_pair = max(stats, key=stats.get)
            
            # Create new token
            new_token_id = len(self.vocab)
            self.merges[most_frequent_pair] = new_token_id
            
            # Build merged token string
            token1_str = self.vocab[most_frequent_pair[0]]
            token2_str = self.vocab[most_frequent_pair[1]]
            merged_str = token1_str + token2_str
            
            # Add to vocabulary
            self.vocab[new_token_id] = merged_str
            
            # Perform the merge
            tokens = self._merge_pair(tokens, most_frequent_pair, new_token_id)
            
            # Progress reporting
            if verbose and (merge_num < 10 or merge_num % 50 == 0):
                print(f"Merge {merge_num + 1:3d}: {repr(token1_str):10s} + {repr(token2_str):10s} "
                      f"= {repr(merged_str):15s} (freq: {stats[most_frequent_pair]:4d}, "
                      f"tokens: {len(tokens):,})")
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Training Complete!")
            print(f"{'='*60}")
            print(f"Final vocabulary size: {len(self.vocab):,}")
            print(f"Final token count: {len(tokens):,}")
            print(f"Compression ratio: {len(text) / len(tokens):.2f}x")
            print(f"{'='*60}\n")
    
    def encode(self, text):
        """
        Convert text to token IDs using learned merges.
        
        Args:
            text: String to encode
            
        Returns:
            List of token IDs
        """
        # Start with character-level encoding
        tokens = []
        for ch in text:
            if ch in self.char_to_id:
                tokens.append(self.char_to_id[ch])
            else:
                # Handle unknown characters - use first character as fallback
                tokens.append(0)
        
        # Apply learned merges in order
        for pair, new_id in self.merges.items():
            tokens = self._merge_pair(tokens, pair, new_id)
        
        return tokens
    
    def decode(self, token_ids):
        """
        Convert token IDs back to text.
        
        Args:
            token_ids: List of token IDs
            
        Returns:
            Decoded string
        """
        tokens = [self.vocab.get(tid, '') for tid in token_ids]
        return ''.join(tokens)
    
    def show_vocab(self, n=20):
        """
        Display vocabulary items.
        
        Args:
            n: Number of items to show
        """
        print(f"\nVocabulary (showing {min(n, len(self.vocab))} of {len(self.vocab)}):")
        print(f"{'ID':<6} {'Token':<20} {'Length':<8}")
        print(f"{'-'*34}")
        for token_id, token_str in sorted(self.vocab.items())[:n]:
            display_str = repr(token_str) if len(token_str) <= 15 else repr(token_str[:12] + '...')
            print(f"{token_id:<6} {display_str:<20} {len(token_str):<8}")


def demo_basic():
    """Basic demonstration of the tokenizer."""
    print("\n" + "="*60)
    print("DEMO 1: Basic Tokenization")
    print("="*60 + "\n")
    
    # Simple repeated text
    text = "low low low lower lowest " * 3
    
    tokenizer = SimpleBPETokenizer(vocab_size=50)
    tokenizer.train(text, verbose=True)
    
    # Show learned vocabulary
    tokenizer.show_vocab(20)
    
    # Test encoding
    test_texts = ["low", "lower", "lowest"]
    print("\nEncoding Examples:")
    print(f"{'Text':<15} {'Token IDs':<20} {'Tokens'}")
    print("-" * 60)
    for test_text in test_texts:
        token_ids = tokenizer.encode(test_text)
        token_strs = [tokenizer.vocab[tid] for tid in token_ids]
        print(f"{test_text:<15} {str(token_ids):<20} {token_strs}")


def demo_realistic():
    """Demonstration with more realistic text."""
    print("\n" + "="*60)
    print("DEMO 2: Realistic Text")
    print("="*60 + "\n")
    
    # More realistic training data
    text = """
    The quick brown fox jumps over the lazy dog.
    Machine learning is transforming artificial intelligence.
    Natural language processing enables computers to understand text.
    Deep learning models learn patterns from data.
    """ * 20  # Repeat for pattern learning
    
    tokenizer = SimpleBPETokenizer(vocab_size=200)
    tokenizer.train(text, verbose=True)
    
    # Show some vocabulary
    tokenizer.show_vocab(30)
    
    # Test compression
    test_sentence = "The quick brown fox jumps over the lazy dog"
    token_ids = tokenizer.encode(test_sentence)
    
    print("\nCompression Analysis:")
    print(f"Original text: '{test_sentence}'")
    print(f"Character count: {len(test_sentence)}")
    print(f"Token count: {len(token_ids)}")
    print(f"Compression ratio: {len(test_sentence) / len(token_ids):.2f}x")
    print(f"\nToken breakdown:")
    for tid in token_ids:
        print(f"  {tid:3d}: {repr(tokenizer.vocab[tid])}")


def demo_comparison():
    """Compare different vocabulary sizes."""
    print("\n" + "="*60)
    print("DEMO 3: Vocabulary Size Comparison")
    print("="*60 + "\n")
    
    text = "The quick brown fox jumps over the lazy dog. " * 50
    test_text = "The quick brown fox"
    
    vocab_sizes = [50, 100, 200]
    
    print(f"Training text length: {len(text)} characters")
    print(f"Test text: '{test_text}'\n")
    print(f"{'Vocab Size':<12} {'Final Tokens':<15} {'Test Compression':<20}")
    print("-" * 50)
    
    for vocab_size in vocab_sizes:
        tokenizer = SimpleBPETokenizer(vocab_size=vocab_size)
        tokenizer.train(text, verbose=False)
        
        test_tokens = tokenizer.encode(test_text)
        compression = len(test_text) / len(test_tokens)
        
        print(f"{vocab_size:<12} {len(tokenizer.vocab):<15} {compression:.2f}x ({len(test_tokens)} tokens)")


if __name__ == "__main__":
    # Run all demonstrations
    demo_basic()
    demo_realistic()
    demo_comparison()
    
    print("\n" + "="*60)
    print("Try It Yourself!")
    print("="*60)
    print("\nExperiment ideas:")
    print("1. Train on code instead of natural language")
    print("2. Try different vocabulary sizes (10, 100, 1000)")
    print("3. Compare compression on similar vs different text")
    print("4. Add special tokens like <START> and <END>")
    print("\nExample usage:")
    print("  tokenizer = SimpleBPETokenizer(vocab_size=100)")
    print("  tokenizer.train(your_text)")
    print("  tokens = tokenizer.encode('test text')")
    print("  decoded = tokenizer.decode(tokens)")


DEMO 1: Basic Tokenization

Training BPE Tokenizer
Starting vocabulary: 8 characters
Target vocabulary: 50 tokens
Will perform 42 merges
Text length: 75 characters → 75 initial tokens

Merge   1: 'l'        + 'o'        = 'lo'            (freq:   15, tokens: 60)
Merge   2: 'lo'       + 'w'        = 'low'           (freq:   15, tokens: 45)
Merge   3: ' '        + 'low'      = ' low'          (freq:   14, tokens: 31)
Merge   4: ' low'     + ' low'     = ' low low'      (freq:    8, tokens: 26)
Merge   5: ' low'     + 'e'        = ' lowe'         (freq:    4, tokens: 22)
Merge   6: 'r'        + ' lowe'    = 'r lowe'        (freq:    3, tokens: 19)
Merge   7: 'r lowe'   + 's'        = 'r lowes'       (freq:    3, tokens: 16)
Merge   8: 'r lowes'  + 't'        = 'r lowest'      (freq:    3, tokens: 13)
Merge   9: 'r lowest' + ' low low' = 'r lowest low low' (freq:    2, tokens: 11)
Merge  10: 'r lowest low low' + ' low low' = 'r lowest low low low low' (freq:    2, tokens: 9)

No more pair